# Figure S5: ABC Atlas Gene Expression Comparison

This notebook compares STARR-FISH gene expression data with the Allen Brain Cell (ABC) Atlas to validate cell type annotations.

## Overview
- **Goal**: Validate STARR-FISH cell type labels by comparing with ABC Atlas reference
- **ABC Atlas**: Comprehensive mouse whole brain single-cell RNA-seq atlas
- **Approach**: 
  1. Load ABC Atlas 10Xv2 and 10Xv3 data
  2. Generate pseudobulk profiles by cell type
  3. Compare expression correlations between STARR-FISH and ABC Atlas
  4. Perform label transfer to assess annotation consistency

## Contents
1. Setup and imports
2. ABC Atlas data loading and metadata
3. Data file verification and download
4. Gene list extraction from STARR-FISH data
5. ABC Atlas data aggregation
6. Pseudobulk generation
7. Expression correlation analysis
8. Label transfer via PCA and k-NN
9. Confusion matrix validation

## 1. Setup and Imports

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import anndata
import time
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache, LocalCache
PWD = '/gpfs/commons/groups/ren_lab/guojiezhong/starr-fish/STARRFISH_in_vivo'

## 2. Load ABC Atlas Cache

The ABC Atlas provides comprehensive single-cell RNA-seq data for the mouse whole brain.
We'll use the `AbcProjectCache` to access pre-processed data files.

In [ ]:
download_base = Path('STARRFISH_in_vivo/Data/abc_atlas')
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

## 3. Load Cell Metadata

Load cell-level metadata including:
- Cell labels (barcodes)
- Cluster assignments
- Cell type annotations
- Spatial coordinates

In [ ]:
cell = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print("Number of cells = ", len(cell))
cell.head(5)

## 4. Load Cluster Annotations

Load detailed cluster annotations including:
- Cluster aliases
- Cell class and subclass
- Taxonomic hierarchy

In [ ]:
cluster_details = abc_cache.get_metadata_dataframe(
    directory='WMB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership_pivoted',
    keep_default_na=False
)
cluster_details.set_index('cluster_alias', inplace=True)
cluster_details.head(5)

## 5. Merge Cell Metadata with Annotations

In [ ]:
cell_extended = cell.join(cluster_details, on='cluster_alias')
cell_extended.head(5)

## 6. Verify and Download 10Xv2 Data Files

Check if 10Xv2 data files are available locally. If corrupted or missing, re-download them.

In [ ]:
abc_cache._local = True
for file in abc_cache.list_data_files('WMB-10Xv2'):
    try:
        file = abc_cache.get_data_path(directory='WMB-10Xv2', file_name=file)
        print(file)
    except Exception as e:
        print(f"Error processing {file}: {e}")
        # needs to be redownloaded
        abc_cache._local = False
        file = abc_cache.get_data_path(directory='WMB-10Xv2', file_name=file, force_download=True)
        print(f"Redownloaded {file}")
        abc_cache._local = True

## 7. Verify and Download 10Xv3 Data Files

Check if 10Xv3 data files are available locally. If corrupted or missing, re-download them.

In [ ]:
for file in abc_cache.list_data_files('WMB-10Xv3'):
    try:
        file = abc_cache.get_data_path(directory='WMB-10Xv3', file_name=file)
        print(file)
    except Exception as e:
        print(f"Error processing {file}: {e}")
        # needs to be redownloaded
        abc_cache._local = False
        file = abc_cache.get_data_path(directory='WMB-10Xv3', file_name=file, force_download=True)
        print(f"Redownloaded {file}")
        abc_cache._local = True

## 8. Extract Gene List from STARR-FISH Data

Load STARR-FISH data and extract the list of genes to focus analysis on common genes between datasets.

In [ ]:
adata3 = sc.read_h5ad(f'{PWD}/Data/scdata_5_28_2025_BRBB500gn_final_CRE_T7CRE.h5ad')
adata3.X = adata3.obsm['X_raw'].copy()
sc.pp.normalize_total(adata3, target_sum=1e6)
sc.pp.log1p(adata3)
gene_list = adata3.var_names.tolist()

## 9. Load and Aggregate 10Xv2 Data

Load all 10Xv2 h5ad files and:
1. Filter for genes present in STARR-FISH data
2. Concatenate all batches into a single dataset

In [ ]:
wmb_10xv2_adata = None
for file in abc_cache.list_data_files('WMB-10Xv2'):
    print(file)
    if 'log2' in file:
        continue
    data_path = abc_cache.get_data_path(directory='WMB-10Xv2', file_name=file)
    adata = sc.read_h5ad(data_path)
    adata.var_names_make_unique()
    # filter gene list
    adata = adata[:, adata.var['gene_symbol'].isin(gene_list)]
    adata.obs['batch'] = file
    if wmb_10xv2_adata is None:
        wmb_10xv2_adata = adata
    else:
        wmb_10xv2_adata = wmb_10xv2_adata.concatenate(adata, index_unique=None)
    del adata

## 10. Load and Aggregate 10Xv3 Data

Load all 10Xv3 h5ad files and:
1. Filter for genes present in STARR-FISH data
2. Concatenate all batches into a single dataset

In [ ]:
wmb_10xv3_adata = None
for file in abc_cache.list_data_files('WMB-10Xv3'):
    print(file)
    if 'log2' in file:
        continue
    data_path = abc_cache.get_data_path(directory='WMB-10Xv3', file_name=file)
    adata = sc.read_h5ad(data_path)
    adata.var_names_make_unique()
    # filter gene list
    adata = adata[:, adata.var['gene_symbol'].isin(gene_list)]
    adata.obs['batch'] = file
    if wmb_10xv3_adata is None:
        wmb_10xv3_adata = adata
    else:
        wmb_10xv3_adata = wmb_10xv3_adata.concatenate(adata, index_unique=None)
    del adata

## 11. Add Cell Type Labels to ABC Atlas Data

Join cell metadata with the loaded expression data to add cell type annotations.

In [ ]:
columns_to_drop = ['cell_barcode', 'library_label']
cell_extended_filtered = cell_extended.drop(columns=columns_to_drop, errors='ignore')
wmb_10xv2_adata.obs = wmb_10xv2_adata.obs.join(
    cell_extended_filtered,
    on=['cell_label'], how='left'
)
wmb_10xv3_adata.obs = wmb_10xv3_adata.obs.join(
    cell_extended_filtered,
    on=['cell_label'], how='left'
)

## 12. Define Pseudobulk Generation Function

Create a function to generate pseudobulk profiles by aggregating single-cell counts within each cell type.

In [ ]:
def generate_pseudobulk(adata, groupby):
    pseudobulk_dict = {}
    groups = adata.obs[groupby].unique()
    # remove nan groups
    groups = groups[~pd.isna(groups)]
    for group in groups:
        print(f"Processing group: {group}")
        group_adata = adata[adata.obs[groupby] == group]
        pseudobulk_counts = group_adata.X.sum(axis=0)
        pseudobulk_dict[group] = np.asarray(pseudobulk_counts).flatten()
    pseudobulk_df = pd.DataFrame.from_dict(
        pseudobulk_dict,
        orient='index',
        columns=adata.var_names
    )
    return pseudobulk_df

## 13. Concatenate 10Xv2 and 10Xv3 Data

Combine both datasets and save for future analysis.

In [ ]:
wmb = wmb_10xv2_adata.concatenate(wmb_10xv3_adata, index_unique=None)
# save
wmb.write_h5ad(f'{PWD}/Data/abc_atlas/WMB_10xv2v3_combined.h5ad')
wmb_10xv2_adata.write_h5ad(f'{PWD}/Data/abc_atlas/WMB_10xv2.h5ad')
wmb_10xv3_adata.write_h5ad(f'{PWD}/Data/abc_atlas/WMB_10xv3.h5ad')

## 14. Load Saved Data (Optional)

If data was previously processed and saved, load it directly.

In [ ]:
wmb = sc.read_h5ad(f'{PWD}/Data/abc_atlas/WMB_10xv2v3_combined.h5ad')
wmb_10xv2_adata = sc.read_h5ad(f'{PWD}/Data/abc_atlas/WMB_10xv2.h5ad')
wmb_10xv3_adata = sc.read_h5ad(f'{PWD}/Data/abc_atlas/WMB_10xv3.h5ad')

## 15. Generate Pseudobulk Profiles

Create pseudobulk expression profiles for:
- Combined ABC Atlas data
- 10Xv2 data only
- 10Xv3 data only
- STARR-FISH data

In [ ]:
wmb_pseudobulk = generate_pseudobulk(wmb, groupby='subclass')
wmb_pseudobulk.columns = wmb.var['gene_symbol']
wmb_10xv2_pseudobulk = generate_pseudobulk(wmb_10xv2_adata, groupby='subclass')
wmb_10xv2_pseudobulk.columns = wmb_10xv2_adata.var['gene_symbol']
wmb_10xv3_pseudobulk = generate_pseudobulk(wmb_10xv3_adata, groupby='subclass')
wmb_10xv3_pseudobulk.columns = wmb_10xv3_adata.var['gene_symbol']
adata3.X = adata3.obsm['X_raw']
starrfish_pseudobulk = generate_pseudobulk(adata3, groupby='subclass_name')
starrfish_pseudobulk = starrfish_pseudobulk[wmb_10xv2_pseudobulk.columns]

## 16. Log TPM Normalization

Normalize pseudobulk data using TPM (Transcripts Per Million) followed by log transformation:
- TPM normalization: accounts for sequencing depth differences
- Log transformation: stabilizes variance

In [ ]:
def log_tpm_transform(df):
    tpm = df.div(df.sum(axis=1), axis=0) * 1e6
    log_tpm = np.log1p(tpm)
    return log_tpm

wmb_pseudobulk_logtpm = log_tpm_transform(wmb_pseudobulk)
starrfish_pseudobulk_logtpm = log_tpm_transform(starrfish_pseudobulk)

## 17. Compute Expression Correlations

Calculate pairwise Pearson correlations between ABC Atlas and STARR-FISH pseudobulk profiles.
- Filter to cell types with ≥100 cells in STARR-FISH
- Higher correlations indicate better agreement between datasets

In [ ]:
# raw correlation of wmb_pseudobulk vs starrfish_pseudobulk, row by row
# drop the cell types with < 100 cells in starrfish_pseudobulk_logtpm
celltype_n = adata3.obs['subclass_name'].value_counts()
correlation_matrix = pd.DataFrame(index=celltype_n.index[celltype_n >= 100],
                                  columns=celltype_n.index[celltype_n >= 100])
for subclass in correlation_matrix.index:
    for starrfish_subclass in correlation_matrix.columns:
        corr = np.corrcoef(
            wmb_pseudobulk_logtpm.loc[subclass],
            starrfish_pseudobulk_logtpm.loc[starrfish_subclass]
        )[0, 1]
        correlation_matrix.loc[subclass, starrfish_subclass] = corr
# reorder rows and columns
correlation_matrix = correlation_matrix.reindex(
    index=sorted(correlation_matrix.index),
    columns=sorted(correlation_matrix.columns)
)

## 18. Plot Correlation Heatmap (Figure S5a)

Visualize correlations between ABC Atlas and STARR-FISH cell types.
- Diagonal elements: same cell type correlation
- Off-diagonal: cross-cell type correlations
- Strong diagonal signal indicates good annotation consistency

In [ ]:
fig, ax = plt.subplots(figsize=(25, 20))
sns.heatmap(correlation_matrix.astype(float), annot=False, cmap='coolwarm', ax=ax)
ax.set_title('Correlation between WMB Pseudobulk and STARRFISH Pseudobulk')
ax.set_xlabel('STARRFISH Subclass')
ax.set_ylabel('WMB Subclass')
fig.tight_layout()
fig.savefig(f'{PWD}/results/expr3/abc_atlas/WMB_vs_STARRFISH_pseudobulk_correlation_heatmap.pdf')

## 19. Prepare ABC Atlas for Label Transfer

Perform PCA and compute k-nearest neighbors on ABC Atlas data to prepare for label transfer:
1. Normalize and log-transform
2. Dimensionality reduction via PCA
3. Compute neighbor graph

In [ ]:
# first do pca and neighbors on wmb
sc.pp.normalize_total(wmb, target_sum=1e6)
sc.pp.log1p(wmb)
sc.tl.pca(wmb)
sc.pp.neighbors(wmb)

## 20. Perform Label Transfer via Ingest

Use scanpy's `ingest` function to:
1. Map STARR-FISH cells into ABC Atlas PCA space
2. Transfer UMAP coordinates
3. Prepare for k-NN classification

In [ ]:
# rename the adata3 var names to gene ID
wmb.var['gene_ID'] = wmb.var_names.copy()
# add the umap coords to wmb.obsm
wmb_umap = np.array((wmb.obs['x'], wmb.obs['y'])).T
wmb.obsm['X_umap'] = wmb_umap
# add fake umap params
wmb.uns['umap'] = {}
wmb.uns['umap']['params'] = {'a': 1.0, 'b': 1.0}
# map gene symbols to gene IDs
adata3.var_names = wmb.var['gene_ID'].groupby(wmb.var['gene_symbol']).first().reindex(adata3.var_names).values
# reorder adata3 var to match wmb var
adata3 = adata3[:, wmb.var['gene_ID'].values]
sc.tl.ingest(adata3, wmb, embedding_method='pca')

## 21. Label Transfer via k-NN Classification

Use k-nearest neighbors to predict ABC Atlas cell type labels for STARR-FISH cells:
- Train k-NN on ABC Atlas PCA embeddings
- Predict labels for STARR-FISH cells in the same space
- Compare predicted vs. original annotations

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Get the PCA representation used for neighbors in wmb
wmb_rep = wmb.obsm['X_pca']

# Get query representation (already computed by ingest)
query_rep = adata3.obsm['X_pca']

# Train k-NN classifier on reference labels
k = wmb.uns['neighbors']['params']['n_neighbors']
knn = KNeighborsClassifier(n_neighbors=k)
# drop wmb obs with nan
knn.fit(wmb_rep[~wmb.obs['subclass'].isna()], wmb.obs.loc[~wmb.obs['subclass'].isna(), 'subclass'])

# Predict labels for query data
adata3.obs['subclass'] = knn.predict(query_rep)
adata3.obs['subclass'] = adata3.obs['subclass'].astype('category')

## 22. Create Label Transfer Confusion Matrix

Generate confusion matrix to assess label transfer accuracy:
- Rows: Original STARR-FISH annotations
- Columns: Predicted ABC Atlas annotations
- Diagonal values: agreement between datasets
- Normalized by row (original annotation)

In [ ]:
# create confusion matrix
confusion_matrix = pd.crosstab(
    adata3.obs['subclass_name'],
    adata3.obs['subclass'],
    normalize='index'
)
# reorder rows and columns
confusion_matrix = confusion_matrix.reindex(
    index=sorted(confusion_matrix.index),
    columns=sorted(confusion_matrix.columns)
)
# filter out cells with < 100 original cells
celltype_n = adata3.obs['subclass_name'].value_counts()
confusion_matrix = confusion_matrix.loc[celltype_n.index[celltype_n >= 100],
                                        celltype_n.index[celltype_n >= 100]]

## 23. Plot Confusion Matrix Heatmap

Visualize label transfer results:
- Strong diagonal indicates accurate cell type annotations
- Off-diagonal entries show misclassifications or related cell types

In [ ]:
fig, ax = plt.subplots(figsize=(25, 20))
sns.heatmap(confusion_matrix, annot=False, cmap='Reds', ax=ax)
ax.set_title('Label Transfer Confusion Matrix: STARRFISH vs WMB')
ax.set_xlabel('Predicted WMB Subclass')
ax.set_ylabel('Original STARRFISH Subclass')
fig.tight_layout()
fig.savefig(f'{PWD}/results/expr3/abc_atlas/WMB_vs_STARRFISH_label_transfer_confusion_heatmap.pdf')

## 24. Save Integrated Data

Save STARR-FISH data with transferred ABC Atlas labels for downstream analysis.

In [ ]:
adata3.write_h5ad(f'{PWD}/Data/scdata_5_28_2025_BRBB500gn_final_CRE_T7CRE_integrate_wmb.h5ad')

## 25. Label Transfer Accuracy vs Cell Type Size

Examine relationship between cell type abundance and label transfer accuracy:
- Diagonal values from confusion matrix = transfer accuracy
- Cell type size = number of cells in original STARR-FISH data
- Expectation: larger cell types should have more accurate labels

In [ ]:
confusion_diag = confusion_matrix.values.diagonal()
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(x=celltype_n[celltype_n >= 100], y=confusion_diag, ax=ax)
ax.set_title('Label Transfer Accuracy vs Original Cell Type Size')
ax.set_xlabel('Original Cell Type Size (Number of Cells)')
ax.set_ylabel('Label Transfer Accuracy (Diagonal of Confusion Matrix)')
ax.set_xscale('log')
fig.tight_layout()
fig.savefig(f'{PWD}/results/expr3/abc_atlas/WMB_vs_STARRFISH_label_transfer_accuracy_vs_size.pdf')

## Conclusions

This analysis validates STARR-FISH cell type annotations by:
1. **Expression correlation**: Comparing pseudobulk gene expression profiles with ABC Atlas
2. **Label transfer**: Using k-NN to predict ABC Atlas labels for STARR-FISH cells
3. **Confusion matrix**: Quantifying agreement between original and transferred labels

High diagonal values in both correlation and confusion matrices indicate:
- Accurate cell type annotations in STARR-FISH data
- Consistent gene expression patterns with reference atlas
- Reliable cell type identity assignments